# Outlier Handling and KDE Analysis

This notebook performs outlier diagnostics separately from preprocessing, and produces outlier QA artifacts.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory")

ROOT = resolve_repo_root(Path.cwd())
INPUT_PATH = ROOT / "data" / "interim" / "ILPD_cleaned.csv"
REPORTS_DIR = ROOT / "produced_reports" / "docs"
FIGURES_DIR = ROOT / "produced_reports" / "figures" / "eda" / "outliers"

for directory in [REPORTS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_PATH}")
print(f"Reports: {REPORTS_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 1) Load cleaned dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)
print(df.shape)
df.head()

## 2) Outlier detection report (IQR, Z-score, Modified Z-score)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop(["Result", "Gender"], errors="ignore")

report_rows = []
for col in num_cols:
    series = df[col]

    # IQR method
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    iqr_lower = q1 - 1.5 * iqr
    iqr_upper = q3 + 1.5 * iqr
    iqr_outliers = int(((series < iqr_lower) | (series > iqr_upper)).sum())

    # Z-score method
    std = series.std(ddof=0)
    mean = series.mean()
    if std == 0:
        z_outliers = 0
    else:
        z = ((series - mean) / std).abs()
        z_outliers = int((z > 3).sum())

    # Modified Z-score method
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        mz_outliers = 0
    else:
        modified_z = (0.6745 * (series - median) / mad).abs()
        mz_outliers = int((modified_z > 3.5).sum())

    report_rows.append({
        "feature": col,
        "rows": int(len(df)),
        "iqr_outliers": iqr_outliers,
        "iqr_pct": round(100 * iqr_outliers / len(df), 2),
        "zscore_outliers": z_outliers,
        "zscore_pct": round(100 * z_outliers / len(df), 2),
        "modified_z_outliers": mz_outliers,
        "modified_z_pct": round(100 * mz_outliers / len(df), 2),
    })

outlier_report = pd.DataFrame(report_rows).sort_values("iqr_pct", ascending=False)
outlier_report

## 3) Kernel Density Estimation (KDE) visuals

In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df, x=col, fill=True)
    plt.title(f"KDE - {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"kde_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 4) Persist outlier artifacts

In [ ]:
outlier_report_path = REPORTS_DIR / "ILPD_outlier_report.csv"
outlier_report.to_csv(outlier_report_path, index=False)

print("Saved:")
print(outlier_report_path)
print(FIGURES_DIR)

## Export notebook to HTML report

In [ ]:
import subprocess

HTML_DIR = ROOT / "produced_reports" / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = ROOT / "scripts" / "outlier_handling.ipynb"

subprocess.run([
    "jupyter", "nbconvert",
    "--to", "html",
    str(NOTEBOOK_PATH),
    "--output-dir", str(HTML_DIR),
], check=True)

print(f"Exported HTML to: {HTML_DIR / NOTEBOOK_PATH.with_suffix('.html').name}")